使用先前检查点的配置调用图以从该点重放。

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, NotRequired
from langchain_core.utils.uuid import uuid7

class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]


def generate_topic(state: State):
    return {"topic": "socks in the dryer"}


def write_joke(state: State):
    return {"joke": f"Why do {state['topic']} disappear? They elope!"} # type: ignore


checkpointer = InMemorySaver()
graph = (
    StateGraph(State)
    .add_node("generate_topic", generate_topic)
    .add_node("write_joke", write_joke)
    .add_edge(START, "generate_topic")
    .add_edge("generate_topic", "write_joke")
    .add_edge("write_joke", END)
    .compile(checkpointer=checkpointer)
)

In [2]:
# Step 1: Run the graph
config = {"configurable": {"thread_id": str(uuid7())}}
result = graph.invoke({}, config) # type: ignore
result

{'topic': 'socks in the dryer',
 'joke': 'Why do socks in the dryer disappear? They elope!'}

In [3]:
# Step 2: Find a checkpoint to replay from
history = list(graph.get_state_history(config)) # type: ignore
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}") # type: ignore

next=(), checkpoint_id=1f1462a4-8040-6b22-8002-39f21546ded2
next=('write_joke',), checkpoint_id=1f1462a4-803e-6e44-8001-addc6862b308
next=('generate_topic',), checkpoint_id=1f1462a4-803d-6774-8000-1d4f4d157b7f
next=('__start__',), checkpoint_id=1f1462a4-803c-6284-bfff-fe79633f7e44


In [4]:
# Step 3: Replay from a specific checkpoint
# Find the checkpoint before write_joke
before_joke = next(s for s in history if s.next == ("write_joke",))
before_joke.values, before_joke.config

({'topic': 'socks in the dryer'},
 {'configurable': {'thread_id': '019de8dc-e3cb-7a43-b559-501438842599',
   'checkpoint_ns': '',
   'checkpoint_id': '1f1462a4-803e-6e44-8001-addc6862b308'}})

In [5]:
replay_result = graph.invoke(None, before_joke.config)
replay_result

{'topic': 'socks in the dryer',
 'joke': 'Why do socks in the dryer disappear? They elope!'}

分支从过去的一个检查点创建一个带有更改后状态的新分支

In [6]:
# Fork: update state to change the topic
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
)
graph.get_state(fork_config).values

{'topic': 'chickens'}

In [7]:
# Resume from the fork — write_joke re-executes with the new topic
fork_result = graph.invoke(None, fork_config)
fork_result

{'topic': 'chickens', 'joke': 'Why do chickens disappear? They elope!'}

当调用 update_state 时，values 会使用指定 node 的写入器（包括归约器）来应用。

在以下情况下需要显式指定 as_node ：

- 多个节点在同一步更新了状态，而 LangGraph 无法确定哪一个是最后的（ InvalidUpdateError ）。
- 没有执行历史：在一个新的线程上设置状态（在测试中常见）。
- 跳过节点：将 as_node 设置为稍后的节点，以让图认为该节点已经运行。

In [8]:
before_joke.values

{'topic': 'socks in the dryer'}

In [9]:
# Treat this update as if generate_topic produced it.
# Execution resumes at write_joke (the successor of generate_topic).
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "chickens"},
    as_node="generate_topic",
)
graph.get_state(fork_config).values

{'topic': 'chickens'}